In [62]:
# MBE in reward function
# Cell 2: Imports
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_TOKEN"] = "hf-....."

import re
import torch
from datasets import load_dataset
# from trl import GRPOTrainer, GRPOConfig
from src.mbe import mbe_reverse_gram

In [63]:
# Cell 3: Load and prepare GSM8K dataset
dataset = load_dataset("openai/gsm8k", "main")

def extract_gold_answer(answer_text: str) -> str:
    """Extract the numeric answer after ####."""
    match = re.search(r"####\s*(.+)", answer_text)
    if match:
        return match.group(1).strip().replace(",", "")
    return ""

def format_example(example):
    """Convert to chat-style prompt and attach gold answer."""
    example["prompt"] = [{"role": "user", "content": example["question"]}]
    example["gold_answer"] = extract_gold_answer(example["answer"])
    return example

train_dataset = dataset["train"].map(format_example)
test_dataset = dataset["test"].map(format_example)

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")
print(f"Example prompt: {train_dataset[0]['prompt']}")
print(f"Gold answer: {train_dataset[0]['gold_answer']}")

Train: 7473, Test: 1319
Example prompt: [{'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}]
Gold answer: 72


In [64]:
# Cell 4: Define reward functions

def extract_answer_from_completion(text: str) -> str:
    """Parse the final numeric answer from a model completion."""
    # Look for #### pattern first
    match = re.search(r"####\s*([\d,\.\-]+)", text)
    if match:
        return match.group(1).strip().replace(",", "")
    # Fallback: last number in the text
    numbers = re.findall(r"-?[\d,]+\.?\d*", text)
    if numbers:
        return numbers[-1].replace(",", "")
    return ""

def correctness_reward(completions: list[list[dict]], gold_answer: list[str], **kwargs) -> list[float]:
    """Award +1.0 if the model's final numeric answer matches the gold answer."""
    rewards = []
    for completion, gold in zip(completions, gold_answer):
        text = completion[0]["content"]
        predicted = extract_answer_from_completion(text)
        try:
            correct = float(predicted) == float(gold)
        except (ValueError, TypeError):
            correct = False
        rewards.append(1.0 if correct else 0.0)
    return rewards

def format_reward(completions: list[list[dict]], **kwargs) -> list[float]:
    """Award +0.5 if the response contains #### <number> pattern."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"]
        has_format = bool(re.search(r"####\s*[\d,\.\-]+", text))
        rewards.append(0.5 if has_format else 0.0)
    return rewards

# Quick sanity check
test_comp = [[{"content": "The answer is 2+3=5. #### 5"}]]
print("Correctness:", correctness_reward(test_comp, gold_answer=["5"]))
print("Format:", format_reward(test_comp))

Correctness: [1.0]
Format: [0.5]


In [65]:
@torch.no_grad()
def full_forward(model, input_ids):
    """Single forward pass returning logits and all hidden states."""
    outputs = model(input_ids, output_hidden_states=True, use_cache=False)
    return outputs.logits, outputs.hidden_states

def compute_mbe_trace(hidden_states, prompt_len, patch_size=8, layer=-1):
    h = hidden_states[layer][0, prompt_len:, :]  # (T_comp, D)
    T, D = h.shape
    usable = (T // patch_size) * patch_size
    if usable == 0:
        return torch.tensor([0.0])
    h = h[:usable].reshape(-1, patch_size, D)
    mbe_vals = mbe_reverse_gram(h)
    return mbe_vals

def compute_per_layer_mbe(hidden_states, prompt_len, patch_size=8):
    per_layer = []
    for layer_idx in range(1, len(hidden_states)):
        mbe_vals = compute_mbe_trace(hidden_states, prompt_len, patch_size=patch_size, layer=layer_idx)
        per_layer.append(mbe_vals.mean().item())
    return per_layer


def compute_single_completion_mbe(hidden_states_layer, prompt_len):
    """Compute MBE on the full completion hidden states for a single sequence (no patching)."""
    h = hidden_states_layer[0, prompt_len:, :]  # (T_comp, D)
    if h.shape[0] < 2:
        return 0.0
    mbe_val = mbe_reverse_gram(h.unsqueeze(0))  # (1,)
    return mbe_val.item()


class MBEReward:
    """
    MBE-based reward compatible with TRL's reward function interface.
    Runs a forward pass through the model, computes per-layer MBE on
    completion hidden states, returns mean MBE as a scalar reward.

    Usage:
        mbe_reward = MBEReward(tokenizer)
        trainer = GRPOTrainer(model="Qwen/Qwen3-0.6B", reward_funcs=[..., mbe_reward], ...)
        mbe_reward.set_model(trainer.model)  # grab model ref after trainer loads it
        trainer.train()
    """

    def __init__(self, tokenizer, layers=None, use_patch_mbe=False, patch_size=8):
        self.__name__ = "mbe_reward"
        self.model = None  # deferred — set via set_model() after trainer init
        self.tokenizer = tokenizer
        self.layers = layers  # None = all layers (1..N)
        self.use_patch_mbe = use_patch_mbe
        self.patch_size = patch_size

    def set_model(self, model):
        """Bind the model reference (call after GRPOTrainer loads the model)."""
        self.model = model

    @torch.no_grad()
    def __call__(self, prompts, completions, **kwargs) -> list[float]:
        if self.model is None:
            return [0.0] * len(completions)

        rewards = []
        device = next(self.model.parameters()).device

        for prompt, completion in zip(prompts, completions):
            # Reconstruct full text via chat template
            prompt_text = self.tokenizer.apply_chat_template(
                prompt, tokenize=False, add_generation_prompt=True
            )
            completion_text = completion[0]["content"]
            full_text = prompt_text + completion_text

            # Tokenize separately to get accurate prompt_len
            prompt_ids = self.tokenizer(prompt_text, return_tensors="pt")["input_ids"]
            full_ids = self.tokenizer(full_text, return_tensors="pt")["input_ids"].to(device)
            prompt_len = prompt_ids.shape[1]

            comp_len = full_ids.shape[1] - prompt_len
            min_len = self.patch_size if self.use_patch_mbe else 2
            if comp_len < min_len:
                rewards.append(0.0)
                continue

            # Forward pass to get hidden states
            _, hidden_states = full_forward(self.model, full_ids)

            # Compute MBE per layer — patch-based or whole-completion
            n_layers = len(hidden_states)
            layer_indices = self.layers if self.layers is not None else range(1, n_layers)

            if self.use_patch_mbe:
                per_layer = [
                    compute_mbe_trace(hidden_states, prompt_len,
                                      patch_size=self.patch_size, layer=li).mean().item()
                    for li in layer_indices
                ]
            else:
                per_layer = [
                    compute_single_completion_mbe(hidden_states[li], prompt_len)
                    for li in layer_indices
                ]

            # Scalar reward = mean MBE across selected layers
            reward = sum(per_layer) / len(per_layer) if per_layer else 0.0
            rewards.append(reward)

        return rewards

In [66]:
# Idea 0: Correctness-Gated MBE Reward
# ---------------------------------------------------------------
# MBE reward is only applied when the answer is correct.
# Among correct completions, higher MBE → higher reward.
# Incorrect completions → 0.0 (no MBE signal).
# This prevents reward hacking where model maximizes MBE without solving the problem.

class CorrectnessGatedMBEReward:
    """
    Correctness-gated MBE reward: MBE only counts when the answer is correct.

    For each completion:
        - If incorrect → 0.0
        - If correct   → mean MBE across selected layers (whole-completion, no patching)

    This lets MBE differentiate quality among correct completions without
    the model learning to game MBE at the expense of correctness.

    Usage:
        gated_mbe = CorrectnessGatedMBEReward(tokenizer)
        trainer = GRPOTrainer(
            model="Qwen/Qwen3-0.6B",
            reward_funcs=[correctness_reward, format_reward, gated_mbe],
            ...
        )
        gated_mbe.set_model(trainer.model)
        trainer.train()
    """

    def __init__(self, tokenizer, layers=None):
        self.__name__ = "correctness_gated_mbe"
        self.model = None
        self.tokenizer = tokenizer
        self.layers = layers

    def set_model(self, model):
        self.model = model

    @torch.no_grad()
    def __call__(self, prompts, completions, gold_answer=None, **kwargs) -> list[float]:
        if self.model is None or gold_answer is None:
            return [0.0] * len(completions)

        device = next(self.model.parameters()).device
        rewards = []

        for prompt, completion, gold in zip(prompts, completions, gold_answer):
            completion_text = completion[0]["content"]

            # Gate: check correctness first
            predicted = extract_answer_from_completion(completion_text)
            try:
                is_correct = float(predicted) == float(gold)
            except (ValueError, TypeError):
                is_correct = False

            if not is_correct:
                rewards.append(0.0)
                continue

            # Correct answer — compute MBE
            prompt_text = self.tokenizer.apply_chat_template(
                prompt, tokenize=False, add_generation_prompt=True
            )
            full_text = prompt_text + completion_text

            prompt_ids = self.tokenizer(prompt_text, return_tensors="pt")["input_ids"]
            full_ids = self.tokenizer(full_text, return_tensors="pt")["input_ids"].to(device)
            prompt_len = prompt_ids.shape[1]

            comp_len = full_ids.shape[1] - prompt_len
            if comp_len < 2:
                rewards.append(0.0)
                continue

            _, hidden_states = full_forward(self.model, full_ids)

            n_layers = len(hidden_states)
            layer_indices = self.layers if self.layers is not None else range(1, n_layers)
            per_layer = [
                compute_single_completion_mbe(hidden_states[li], prompt_len)
                for li in layer_indices
            ]
            mbe_val = sum(per_layer) / len(per_layer) if per_layer else 0.0
            rewards.append(mbe_val)

        return rewards


# Quick sanity check
print("=== CorrectnessGatedMBEReward sanity check (no model) ===")
_gated = CorrectnessGatedMBEReward(tokenizer=None)
print("No model → all zeros:", _gated(
    prompts=[None, None],
    completions=[[{"content": "#### 5"}], [{"content": "#### 3"}]],
    gold_answer=["5", "5"],
))

=== CorrectnessGatedMBEReward sanity check (no model) ===
No model → all zeros: [0.0, 0.0]


In [ ]:
# Cell 5: Training with Correctness-Gated MBE reward
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Correctness-gated MBE: only rewards MBE when the answer is correct
gated_mbe = CorrectnessGatedMBEReward(tokenizer)

config = GRPOConfig(
    output_dir="gated_mbe_gsm8k_output",
    num_generations=8,
    max_completion_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=1,
    logging_steps=10,
    max_steps=200,
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.3,
    bf16=True,
    save_strategy="no",
    report_to="none",
)

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=[correctness_reward, format_reward, gated_mbe],
    args=config,
    train_dataset=train_dataset,
)

# Bind model ref after trainer loads it
gated_mbe.set_model(trainer.model)

trainer.train()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


TypeError: GRPOTrainer._get_train_sampler() takes 1 positional argument but 2 were given